### Agents to help with trade prematching

#### Import Libraries

In [1]:
from langchain.vectorstores import Chroma
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.document_loaders import YoutubeLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import WebBaseLoader
from langchain.agents import initialize_agent, Tool
from langchain.agents import load_tools
from langchain.agents import AgentType
from langchain.tools import BaseTool
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import TextLoader
from langchain.storage import InMemoryStore
from langchain.embeddings import OpenAIEmbeddings
from langchain.retrievers import ParentDocumentRetriever
from langchain.memory import ConversationBufferMemory
from langchain.document_loaders.csv_loader import CSVLoader

https://medium.com/google-cloud/competitor-analytics-with-langchain-agents-and-vertex-palm-api-410453cecd83

#### Initialise

In [2]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model="gpt-4", temperature=0)

#### Load random document

In [4]:
loaders = [
    TextLoader("./paul_graham_essay.txt")
    #TextLoader("../../state_of_the_union.txt"),
]
docs = []
for l in loaders:
    docs.extend(l.load())

In [5]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# This text splitter is used to create the child documents
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# The storage layer for the parent documents
store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

retriever.add_documents(docs)

search = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever
)

#### Load csv file of our trade details

In [11]:
csv_loaders = CSVLoader(file_path="./trade_details.csv")
csv_docs = []
csv_docs.extend(csv_loaders.load())

csv_db = FAISS.from_documents(csv_docs, OpenAIEmbeddings())

csv_ret = csv_db.as_retriever(search_kwargs={"k": 1})

csv_search = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=csv_ret
)

#### Load emails

In [14]:
email_loaders = TextLoader("./email.txt")
email_docs = []
email_docs.extend(email_loaders.load())

email_store = FAISS.from_documents(email_docs, OpenAIEmbeddings())
email_ret = email_store.as_retriever(search_kwargs={"k": 1})

email_search = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=email_ret
)

#### Define tools for agent

In [15]:
tools = [
    Tool(
        name="paul",
        func=search.run,
        description="useful when answering questions about paul graham."
    ),
    Tool(
        name="trade details",
        func=csv_search.run,
        description="useful when answering questions about our trade details"
    ),
    Tool(
        name="email details",
        func=email_search.run,
        description="useful to find emails about trades from counterparty"
    )
]

#### Initialise agent

In [16]:
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

#### Run with question/task

In [17]:
agent.run("Match the trade details we have with that of the counterpety and figure out why the trade with OCBC on ISIN AB19293829 is not settled?")



> Entering new AgentExecutor chain...
First, I need to check our own trade details for the trade with OCBC on ISIN AB19293829.
Action: trade details
Action Input: OCBC, ISIN AB19293829
Observation: The trade with ISIN AB19293829 and counterparty OCBC has not yet settled as of the current date, 15/10/2020. The trade date was 13/10/20 and the settlement date is also 15/10/20. The settlement amount is 152792.38 and the place of settlement is EC 92416.
Thought:Now that I have our trade details, I need to compare them with the counterparty's trade details.
Action: email details
Action Input: OCBC, ISIN AB19293829
Observation: The ISIN AB19293829 is associated with a trade settlement that occurred on 13/10/20 and was supposed to be settled by 15/10/20. The amount for this trade was 152792.38. However, as of the last update, it was not settled. The EC number for this trade is 25625.
Thought:The trade details from our side and the counterparty's side match in terms of trade date, settlement 

"The trade with OCBC on ISIN AB19293829 has not settled because there is a discrepancy in the place of settlement EC numbers. Our records show the EC number as 92416, while OCBC's records show it as 25625. This needs to be rectified for the trade to settle."

### Creating agents through custom functions

In [19]:
def combine_page_contents(list_of_dicts):
    # This function assumes each dictionary in the list has a key 'page content'
    combined_content = "\n".join(d.page_content for d in list_of_dicts)
    return combined_content

In [20]:
import chromadb
chroma_client = chromadb.Client()

In [21]:
csv_store = chroma_client.create_collection(name="csv_list")

In [22]:
email_store = chroma_client.create_collection(name="email_list")

In [23]:
def extract_page_contents(list_of_dicts):
    # Initialize an empty list to store the page contents
    page_contents = []

    # Iterate through each dictionary in the list
    for dictionary in list_of_dicts:
        # Extract the 'page content' from each dictionary
        content = dictionary.page_content  # Using .get() for safe extraction

        # Append the extracted content to the list
        page_contents.append(content)

    return page_contents


In [24]:
email_loaders = TextLoader("./email.txt")
email_docs = []
email_docs.extend(email_loaders.load())

In [25]:
csv_loaders = CSVLoader(file_path="./trade_details.csv")
csv_docs = []
csv_docs.extend(csv_loaders.load())

In [26]:
csv_list = extract_page_contents(csv_docs)

In [27]:
email_list = extract_page_contents(email_docs)

In [28]:
def generate_ids(list_length, key):
    # Create a list of IDs where each ID is of the format "id" followed by the index number
    ids = [key + str(i) for i in range(1, list_length + 1)]
    return ids

In [29]:
csv_store.add(
    documents=csv_list,
    #metadatas=[{"source": "my_source"}, {"source": "my_source"}],
    ids=generate_ids(len(csv_list), 'csv')
)

In [30]:
email_store.add(
    documents=email_list,
    #metadatas=[{"source": "my_source"}, {"source": "my_source"}],
    ids=generate_ids(len(email_list), 'email')
)

In [33]:
def answer_question(
    trade_query
    ):
    """
    Answer a question based on the most similar context from the dataframe texts
    """
    global email_store
    global csv_store

    email = email_store.query(
    query_texts=[trade_query],
    n_results=1
    )  

    csv =  csv_store.query(
    query_texts=[trade_query],
    n_results=1
    )  

    csv_doc = csv['documents'][0][0]
    email_doc = email['documents'][0][0]
    #email_trade = combine_page_contents(email_doc)
    print('email trade:\n' + email_doc)
    #csv_trade = combine_page_contents(csv_doc)
    print('\ncsv trade:\n' + csv_doc)
    # Create a chat completion using the question and context
    response = llm.predict(f"""
    You are a helpful conversational assistant who helps compare email details and our trade details 
    to find why a trade is not settled. Find if the both the sources are pointing to the same trade,
    if yes, find any mismatches. If it can't be 
    answered based on the context, say \"I don't know\"\n\n",
    "our details: {csv_doc}\n\n---\n\ncounterparty: {email_doc}\nAnswer:
    """
    )

    return response

In [34]:
answer_question('why is OCBC AB14293829 not settled yet?')

email trade:
Hi team,

OCBC trade settlement AB19293829
Below trade is not settled. Our place of settlement: EC 23145

Best,
OCBC

csv trade:
﻿ISIN: AB14293829
trade date: 13/10/20
settlement date: 15/10/20
settlement amount: 152792.38
counterparty: OCBC
current date: 15/10/2020
status: not settled
place of settlement: EC 23145


'The ISIN in our details (AB14293829) does not match with the ISIN in the email from the counterparty (AB19293829). This could be the reason why the trade is not settled. All other details including the place of settlement match.'